# 10. lecke - Mesterséges intelligencia ügynökök a gyakorlatban

Ebben a leckében a Microsoft Agent Framework (Python) segítségével működő MI ügynökök **gyakorlati mintáit** tanulod meg. Tartalmazza:

- **Megfigyelhetőség** — időzítés és naplózás hozzáadása az ügynöki interakciókhoz
- **Értékelés** — egy értékelő ügynök használata a válaszok minőségének pontozásához
- **Költségkezelés** — stratégiák a tokenoptimalizálásra és modellválasztásra

A példa egy **utazási ügynök**, amely segíti a felhasználókat az utazások megtervezésében, felügyelettel és értékeléssel kiegészítve.


## Beállítás


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Gyártási szempontok

Az AI ügynökök prototípusoktól a gyártásig történő áthelyezése gondos figyelmet igényel három alappillérre:

1. **Megfigyelhetőség** — Látnod kell, hogy az ügynök mit csinál, mennyi ideig tart, és mely eszközöket hívja meg. Nyomkövetés és naplózás nélkül a gyártási hibák hibakeresése szinte lehetetlen.

2. **Értékelés** — Az automatizált minőségellenőrzések biztosítják, hogy az ügynök válaszai idővel pontosak, teljesek és hasznosak maradjanak. Egy értékelő ügynök pontozhatja a válaszokat meghatározott kritériumok alapján.

3. **Költségkezelés** — A tokenhasználat közvetlen hatással van a költségekre. Olyan stratégiák, mint a promptoptimalizálás, modellválasztás és gyorsítótárazás segítenek az kiadások kordában tartásában anélkül, hogy a minőség romlana.


## Megfigyelhető ügynök létrehozása

Definiáljuk az utazási eszközöket, és az ügynök hívását időzítéssel vesszük körül, hogy figyelemmel kísérhessük a késleltetést. Éles környezetben az OpenTelemetry-vel vagy hasonló trace-elési háttérrel integrálnánk.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Értékelési minták

Egy gyakori gyártási minta, hogy egy második ügynököt használnak **értékelőként**. Az értékelő értékeli az elsődleges ügynök válaszát előre meghatározott kritériumok alapján, például teljesség, pontosság és hasznosság szerint.

Ez lehetővé teszi:
- Automatikus minőségellenőrzést, mielőtt a válaszok eljutnak a felhasználókhoz
- Regresszióérzékelést, amikor a promptok vagy modellek változnak
- Az ügynök teljesítményének folyamatos nyomon követését az idő során


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Költségkezelési stratégiák

A költségek ellenőrzése kulcsfontosságú a termelő AI ügynökök számára. Íme a fő stratégiák:

| Stratégia | Leírás |
|---|---|
| **Prompt optimalizálás** | Tartsa a rendszerutasításokat tömörek. Távolítsa el a fölösleges kontextust az input tokenek csökkentése érdekében. |
| **Modellválasztás** | Használjon kisebb, olcsóbb modelleket (pl. GPT-4o-mini) egyszerű feladatokhoz, mint osztályozás vagy kivonatolás, és tartsa fenn a nagyobb modelleket összetett érveléshez. |
| **Gyorsítótár** | Gyorsítótárazza az eszközök eredményeit és gyakori lekérdezéseket, hogy elkerülje az ismétlődő API hívásokat. |
| **Token költségvetés** | Állítson be `max_tokens` korlátokat, hogy megakadályozza a váratlanul hosszú válaszokat. |
| **Tömbösítés** | Csoportosítson több felhasználói lekérdezést egyetlen API hívásba, ahol lehetséges. |

Gyakorlatban a rétegezett megközelítés működik jól: az egyszerű kéréseket gyors, olcsó modellhez irányítsa, és csak az összetettebb lekérdezéseket továbbítsa egy jobb képességű modellhez.


## Összefoglaló

Ebben a leckében megtanultad, hogyan:

1. **Adhatsz megfigyelhetőséget** az ügynök interakciókhoz időzítéssel és naplózással, ezzel megalapozva a nyomon követést és a megfigyelést.
2. **Értékelheted az ügynök válaszait** automatikusan egy értékelő ügynök segítségével, amely pontozza a teljességet, pontosságot és hasznosságot.
3. **Kezelheted a költségeket** a prompt optimalizálás, modellválasztás, gyorsítótárazás és token-költségvetés révén.

Ezek a termelési minták segítenek biztosítani, hogy az AI ügynökeid megbízhatóak, mérhetők és költséghatékonyak legyenek nagy léptékben.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
